# First Letters on one Colab GPU: GrowPatch → render → 9 µm ink ensemble → images on Drive

**What this does.** For one eligible scroll and one region you choose, it
1. picks a seed point on the papyrus (from the public Scan-Quality-Map "clearest regions", snapped onto the organisers' m7 surface prediction),
2. grows a surface patch there with **VC3D GrowPatch** (`vc_grow_seg_from_seed`, the tool the PHerc. 306B winners used for their claim surface), headless,
3. renders a 41-layer surface volume around it (`vc_render_tifxyz`, as all three winners did),
4. runs **Hecate 9.6 µm** at several depths and both reading directions (no compiling needed), and optionally the official **ink_9um** seeds 42 + 43 (and Reader V3) through the villa inference CLI,
5. combines the models with the two winning rules (PHerc. 343: average of per-model z-scores; PHerc. 306B: product of two models) and writes ink-over-CT images with a 1 cm scale bar to Google Drive.

**How to use it.** Runtime → Change runtime type → GPU (T4 or L4). Edit the PARAMETERS cell. Then Runtime → Run all.
Everything is written to `/content/fl_work` and the images are copied to `MyDrive/first_letters/`.

**Status of each cell — read this.** This notebook was written on a machine without a GPU and without Colab.
* `SMOKE-TESTED (CPU)` — the code in this cell ran on CPU against the real open-data bucket or synthetic data.
* `SOURCE-VERIFIED, UNTESTED` — every command-line flag was checked against the source code of villa commit `c8f3ba4` (2026-11-20) or the model's own script, but the cell has **never been run**.
* `UNTESTED` — written from documentation or the winners' scripts; expect to fix something.

**Honest limits.** GrowPatch is a CPU program (it uses the GPU not at all); it runs fine headless from the VC3D AppImage, but on Colab it pays GPU-runtime prices while it runs. The official spiral-fitter route (used for PHerc. 343 and 211) needs a native object-store build and much more set-up; it is sketched in Appendix A, untested. A *negative* window is the common outcome — the winners each looked at dozens of windows (PHerc. 0306B: 72 candidate windows on one band of one scroll) before one showed letters.

**Rules reminder.** Do not publish letters you find before the prize is announced; submit through the First Letters form. Never train on, or tune on, the region you will claim.

## 1. Parameters — the only cell you need to edit

In [ ]:
# ==== PARAMETERS ====================================================================================
SCROLL      = "PHerc0175A"      # one of the 19 eligible ids, e.g. PHerc0175A, PHerc0358, PHerc0813, PHerc0826 ...
Z0, Z1      = 6900, 7900        # z-window (level-0 slices) you want to look at; the seed is taken inside it
SEED_XYZ    = None              # None = use the clearest Scan-Quality-Map target inside the window; or (x, y, z)
SQM_TARGET  = 0                 # which SQM target (0, 1, 2) when several lie in the window
GENERATIONS = 100               # GrowPatch growth rings at 20 voxels each (306B winners used 140; ~100 gives ~4-6 cm2)
N_LAYERS    = 41                # render layers (odd; surface = middle layer). 306B used 41, 343 used 66
DEPTHS      = [-2, -1, 0, 1, 2] # depth offsets (native layers) for the ink sweep; 343 swept -15..+15 (expensive)
RUN_POSITIVE_CONTROL = True     # first session only: re-render PHerc.343's winning mesh and check letters appear
RUN_VILLA_INK = False           # True = build villa (20-60 min, UNTESTED) and add ink_9um seeds 42+43 (+ Reader V3)
HECATE_BATCH = 4                # lower to 1-2 if you get CUDA out-of-memory
DRIVE_DIR   = "/content/drive/MyDrive/first_letters"
WORK        = "/content/fl_work"
# =====================================================================================================
RUN_TAG = f"{SCROLL}_z{Z0}-{Z1}"
print(RUN_TAG)

## 2. Google Drive, GPU, disk — `UNTESTED` (Colab-only calls)

In [ ]:
import os, shutil, subprocess, json, time, glob, sys
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("not on Colab: Drive not mounted; outputs stay under", WORK)
os.makedirs(WORK, exist_ok=True); os.makedirs(DRIVE_DIR, exist_ok=True)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout or "NO GPU")
total, used, free = shutil.disk_usage("/content" if os.path.exists("/content") else "/")
print(f"disk free: {free/1e9:.0f} GB  (this notebook needs ~25 GB per window: ~7-11 GB normal grids, 2-6 GB CT cache, 1-2 GB render, 1.3 GB models)")
print("CPU cores:", os.cpu_count())
T0 = time.time(); TIMES = {}
def tick(name):
    TIMES[name] = round((time.time() - T0) / 60, 1); print(f"[{TIMES[name]} min] {name}")

## 3. Helper code and the table of eligible scrolls — `SMOKE-TESTED (CPU)`
The helpers were run on CPU against the real bucket (seed snapping on PHerc0358, CT crop, normal-grid listing and download) and on synthetic renders (resampling, depth crops, both ensemble rules, the claim image). The scroll table lists, for the 19 eligible scrolls, the native volume, m7 surface prediction, normal grids, Lasagna predictions and umbilicus (from a bucket listing on 2026-11-20) and the three clearest regions of the Scan-Quality-Map atlas (github.com/jcooperkai-sys/Scan-Quality-Map, docs/targets.json).

In [ ]:
%%writefile /content/fl_helpers.py
"""fl_helpers.py - small helpers for the First Letters Colab runbook (first_letters_colab.ipynb).

Everything here is plain Python (numpy, zarr<3 or zarr>=3, fsspec, requests, tifffile, opencv, optional torch).
It does NOT run any model or VC3D tool itself; the notebook calls those as shell commands.

Conventions (checked against the winners' repos, see reports/07_winning_recipes.md):
  * coordinates are level-0 voxels, written x, y, z (VC3D order); zarr arrays are (z, y, x)
  * a render made with `vc_render_tifxyz --num-slices N --slice-step 1` has layer k at (k - (N-1)/2) voxels
    along the mesh normal, so with N odd the traced surface is layer (N-1)//2 and depth d = k - (N-1)//2
"""
from __future__ import annotations

import json
import math
import os
import time
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np

BUCKET = "https://vesuvius-challenge-open-data.s3.us-east-1.amazonaws.com"
_NS = {"s": "http://s3.amazonaws.com/doc/2006-03-01/"}


def url(key: str) -> str:
    return f"{BUCKET}/{key.lstrip('/')}"


# ----------------------------------------------------------------------------------------------------------------
# S3 listing / download (anonymous, open-data bucket)
# ----------------------------------------------------------------------------------------------------------------
def s3_ls(prefix: str, recursive: bool = False):
    """(dirs, files[(key, size)]) under prefix. Anonymous ListObjectsV2."""
    import requests
    dirs, files, token = [], [], None
    while True:
        p = {"list-type": "2", "prefix": prefix}
        if not recursive:
            p["delimiter"] = "/"
        if token:
            p["continuation-token"] = token
        r = requests.get(BUCKET, params=p, timeout=60)
        r.raise_for_status()
        t = ET.fromstring(r.content)
        dirs += [e.find("s:Prefix", _NS).text for e in t.findall("s:CommonPrefixes", _NS)]
        files += [(e.find("s:Key", _NS).text, int(e.find("s:Size", _NS).text)) for e in t.findall("s:Contents", _NS)]
        if t.find("s:IsTruncated", _NS).text != "true":
            return dirs, files
        token = t.find("s:NextContinuationToken", _NS).text


def _get(key_size_dest):
    import requests
    key, size, dest = key_size_dest
    if os.path.exists(dest) and os.path.getsize(dest) == size:
        return 0
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    for attempt in range(5):
        try:
            r = requests.get(url(key), timeout=300)
            r.raise_for_status()
            tmp = dest + ".part"
            with open(tmp, "wb") as f:
                f.write(r.content)
            os.replace(tmp, dest)
            return size
        except Exception:  # noqa: BLE001 - retry any network error
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"download failed: {key}")


def download_normal_grids(ngrid_key: str, dest: str, z0: int, z1: int, margin: int = 256, threads: int = 32,
                          max_files: int | None = None, dry_run: bool = False):
    """Copy a normal-grid store (metadata.json + xy/xz/yz *.grid) to a local directory for vc_grow_seg_from_seed.

    xy grids are one file per z slice, so only z in [z0-margin, z1+margin] is fetched; xz and yz grids are one file
    per y / x slice and are fetched whole (about 6 GB for a ~8k-wide scroll). The *_img preview folders are skipped.
    Returns (n_files, n_bytes)."""
    pre = ngrid_key.rstrip("/") + "/"
    jobs = []
    _, top = s3_ls(pre)
    for k, s in top:
        jobs.append((k, s, os.path.join(dest, k[len(pre):])))
    for sub in ("xy/", "xz/", "yz/"):
        _, files = s3_ls(pre + sub)
        for k, s in files:
            if sub == "xy/":
                try:
                    zi = int(os.path.basename(k).split(".")[0])
                except ValueError:
                    continue
                if not (z0 - margin <= zi < z1 + margin):
                    continue
            jobs.append((k, s, os.path.join(dest, k[len(pre):])))
    if max_files is not None:
        jobs = jobs[:max_files]
    total = sum(s for _, s, _ in jobs)
    print(f"normal grids: {len(jobs)} files, {total / 1e9:.2f} GB -> {dest}")
    if dry_run:
        return len(jobs), total
    t0 = time.time()
    with ThreadPoolExecutor(threads) as ex:
        done = sum(ex.map(_get, jobs))
    print(f"  fetched {done / 1e9:.2f} GB new in {time.time() - t0:.0f} s")
    return len(jobs), total


# ----------------------------------------------------------------------------------------------------------------
# remote zarr access (OME-Zarr v2 volumes and predictions in the open-data bucket)
# ----------------------------------------------------------------------------------------------------------------
def open_level(key_or_url: str, level: int | str = 0):
    """Open one pyramid level of a remote zarr v2 group read-only (works with zarr 2.x and 3.x)."""
    import fsspec
    import zarr
    u = key_or_url if key_or_url.startswith("http") else url(key_or_url)
    u = u.rstrip("/") + f"/{level}"
    if int(zarr.__version__.split(".")[0]) >= 3:
        return zarr.open_array(u, mode="r", zarr_format=2)
    return zarr.open(fsspec.get_mapper(u), mode="r")


def read_box(arr, xyz, half: int):
    """(z,y,x) crop of half-size `half` around xyz, clipped to the array; returns (block, origin_zyx)."""
    x, y, z = [int(round(v)) for v in xyz]
    lo = [max(0, c - half) for c in (z, y, x)]
    hi = [min(n, c + half + 1) for c, n in zip((z, y, x), arr.shape)]
    return np.asarray(arr[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]), lo


def snap_seed(m7_key: str, xyz, half: int = 48):
    """Nearest voxel of the (binary, thresholded) m7 surface prediction to xyz, at level 0.
    GrowPatch seeds must lie on the predicted sheet. Returns (x, y, z, distance_voxels) or None."""
    arr = open_level(m7_key, 0)
    blk, (oz, oy, ox) = read_box(arr, xyz, half)
    zz, yy, xx = np.nonzero(blk > 0)
    if len(zz) == 0:
        return None
    x, y, z = xyz
    d2 = (xx + ox - x) ** 2 + (yy + oy - y) ** 2 + (zz + oz - z) ** 2
    i = int(np.argmin(d2))
    return int(xx[i] + ox), int(yy[i] + oy), int(zz[i] + oz), float(math.sqrt(d2[i]))


def sheet_profile(ct_key: str, xyz, half: int = 32):
    """Quick CT look at a seed: mean intensity and the share of dark (gap) voxels in a small cube.
    Not a quality score (use the Scan-Quality-Map atlas for that), just a sanity check that the seed is on papyrus."""
    arr = open_level(ct_key, 0)
    blk, _ = read_box(arr, xyz, half)
    b = blk.astype(np.float32)
    med = float(np.median(b))
    return {"mean": float(b.mean()), "median": med, "dark_fraction": float((b < 0.6 * med).mean()) if med > 0 else None,
            "shape": list(blk.shape)}


# ----------------------------------------------------------------------------------------------------------------
# GrowPatch parameters (keys read by vc_grow_seg_from_seed / GrowPatch.cpp in villa c8f3ba4)
# ----------------------------------------------------------------------------------------------------------------
def write_grow_params(path: str, normal_grid_dir: str, voxel_um: float, generations: int = 140, step: int = 20,
                      threads: int = 4, cache_bytes: int = 4_000_000_000, min_area_cm: float = 0.5):
    """The exact parameter set the PHerc. 0306B winners used for their claim surface
    (discovery/fl/sqm/PHerc0306B_f306_w074own/params.json), with the paths replaced."""
    p = {"mode": "seed", "step_size": step, "min_area_cm": min_area_cm, "generations": generations,
         "thread_limit": threads, "normal_grid_path": str(normal_grid_dir), "voxelsize": voxel_um,
         "cache_size": cache_bytes}
    Path(path).write_text(json.dumps(p, indent=1))
    return p


# ----------------------------------------------------------------------------------------------------------------
# tifxyz
# ----------------------------------------------------------------------------------------------------------------
def tifxyz_stats(mesh_dir: str, umbilicus_json: str | None = None, voxel_um: float | None = None):
    import tifffile
    X = tifffile.imread(os.path.join(mesh_dir, "x.tif")).astype(np.float64)
    Y = tifffile.imread(os.path.join(mesh_dir, "y.tif")).astype(np.float64)
    Z = tifffile.imread(os.path.join(mesh_dir, "z.tif")).astype(np.float64)
    v = (X > 0) & (Y > 0) & (Z > 0)
    out = {"grid_hw": list(X.shape), "valid_nodes": int(v.sum()),
           "x": [float(X[v].min()), float(X[v].max())], "y": [float(Y[v].min()), float(Y[v].max())],
           "z": [float(Z[v].min()), float(Z[v].max())]}
    meta = os.path.join(mesh_dir, "meta.json")
    if os.path.exists(meta):
        m = json.load(open(meta))
        sc = m.get("scale")
        if sc and voxel_um:
            step = 1.0 / float(sc[0] if isinstance(sc, list) else sc)
            out["area_cm2_approx"] = float(v.sum() * (step * voxel_um * 1e-4) ** 2)
    if umbilicus_json:
        cp = sorted(json.load(open(umbilicus_json))["control_points"], key=lambda p: p["z"])
        zc = np.array([p["z"] for p in cp]); xc = np.array([p["x"] for p in cp]); yc = np.array([p["y"] for p in cp])
        r = np.hypot(X[v] - np.interp(Z[v], zc, xc), Y[v] - np.interp(Z[v], zc, yc))
        out["radius_vox_median"] = float(np.median(r))
        if voxel_um:
            out["radius_mm_median"] = float(np.median(r) * voxel_um / 1000)
    return out


# ----------------------------------------------------------------------------------------------------------------
# renders: reading, quick looks, depth windows
# ----------------------------------------------------------------------------------------------------------------
def open_render(path: str):
    """A vc_render_tifxyz --zarr-output store: level '0' is (layers, rows, cols) uint8."""
    import zarr
    g = zarr.open(path, mode="r")
    try:
        return g["0"]
    except Exception:  # noqa: BLE001 - a bare array
        return g


def surface_index(n_layers: int) -> int:
    return (n_layers - 1) // 2


def villa_layer_window(n_layers: int, d: int, depth: int = 17):
    """--layer-start/--layer-end (end exclusive) for villa ink inference: a `depth`-layer window centred on
    surface_index + d (the PHerc. 343 run_ink.sh rule: s = N/2 + d - 8 for N=66)."""
    c = surface_index(n_layers) + d
    s = c - depth // 2
    e = s + depth
    if s < 0 or e > n_layers:
        raise ValueError(f"d={d}: layers {s}..{e - 1} outside 0..{n_layers - 1}")
    return s, e


def quicklook(render, out_dir: str, stride: int = 1):
    """Middle layer and max/mean composites of the central +-5 layers, contrast stretched, as PNGs."""
    import cv2
    os.makedirs(out_dir, exist_ok=True)
    n = render.shape[0]
    c = surface_index(n)
    lo, hi = max(0, c - 5), min(n, c + 6)
    stack = np.asarray(render[lo:hi, ::stride, ::stride]).astype(np.float32)
    outs = {}
    for name, img in (("layer_mid", stack[c - lo]), ("composite_max", stack.max(0)), ("composite_mean", stack.mean(0))):
        p = os.path.join(out_dir, f"{name}.png")
        cv2.imwrite(p, stretch(img))
        outs[name] = p
    return outs


def stretch(img, lo_pct=1.0, hi_pct=99.5, valid=None):
    a = np.asarray(img, dtype=np.float32)
    m = (a > 0) if valid is None else valid
    if not m.any():
        return np.zeros(a.shape, np.uint8)
    lo, hi = np.percentile(a[m], [lo_pct, hi_pct])
    return (np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype(np.uint8)


# ----------------------------------------------------------------------------------------------------------------
# resampling for Hecate (needs 9.6 um isotropic input; hecate.py refuses spacings >2 % off)
# ----------------------------------------------------------------------------------------------------------------
def resample_stack(arr, src_um: float, dst_um: float = 9.6, band: int = 1024, device: str = "cpu"):
    """Trilinear resample of a (Z, Y, X) uint8 stack by src_um/dst_um in all three axes, in row bands.
    Uses torch if available (GPU if device='cuda'), else scipy.ndimage.zoom."""
    f = src_um / dst_um
    Z, Y, X = arr.shape
    oz, oy, ox = max(1, round(Z * f)), max(1, round(Y * f)), max(1, round(X * f))
    out = np.zeros((oz, oy, ox), np.uint8)
    try:
        import torch
        import torch.nn.functional as F
        have_torch = True
    except ImportError:  # pragma: no cover
        have_torch = False
        from scipy import ndimage
    sy = Y / oy                                         # exact source step along Y (align_corners=False)
    for y0o in range(0, oy, band):
        y1o = min(oy, y0o + band)
        yc = (np.arange(y0o, y1o) + 0.5) * sy - 0.5     # source row coordinate of every output row
        yc = np.clip(yc, 0, Y - 1)
        y0 = int(np.floor(yc.min()))
        y1 = min(Y, int(np.floor(yc.max())) + 2)
        src = np.asarray(arr[:, y0:y1, :]).astype(np.float32)
        # 1) Z and X: full extent in this band, so a plain resize is exact; Y kept at native rows
        if have_torch:
            t = torch.from_numpy(src)[None, None].to(device)
            r = F.interpolate(t, size=(oz, y1 - y0, ox), mode="trilinear", align_corners=False)[0, 0].cpu().numpy()
        else:
            r = ndimage.zoom(src, (oz / Z, 1, ox / X), order=1)
        # 2) Y: linear interpolation at the exact source coordinates
        i0 = np.floor(yc).astype(int) - y0
        i1 = np.minimum(i0 + 1, r.shape[1] - 1)
        w = (yc - np.floor(yc)).astype(np.float32)[None, :, None]
        piece = r[:, i0, :] * (1 - w) + r[:, i1, :] * w
        out[:, y0o:y1o, :] = np.clip(np.rint(piece), 0, 255).astype(np.uint8)
    return out


def hecate_crop(res, n_native: int, src_um: float, d_native: float, nz: int = 16, dst_um: float = 9.6):
    """16 planes of a resampled stack centred on the surface + d_native native layers (forward order)."""
    f = src_um / dst_um
    c = (surface_index(n_native) + d_native) * f      # centre in resampled plane units
    s = int(round(c - nz / 2 + 0.5))
    if s < 0 or s + nz > res.shape[0]:
        raise ValueError(f"d={d_native}: planes {s}..{s + nz - 1} outside 0..{res.shape[0] - 1}")
    return np.ascontiguousarray(res[s:s + nz])


# ----------------------------------------------------------------------------------------------------------------
# ensembles (the two winning rules) and the claim-style image
# ----------------------------------------------------------------------------------------------------------------
def load_map(path: str, shape_hw=None):
    """An 8-bit ink map (PNG/TIF) as float32 probability in [0, 1], optionally resized to shape_hw (rows, cols)."""
    import cv2
    import tifffile
    a = tifffile.imread(path) if path.lower().endswith((".tif", ".tiff")) else cv2.imread(path, cv2.IMREAD_UNCHANGED)
    a = np.asarray(a).astype(np.float32)
    if a.ndim == 3:
        a = a[..., 0]
    a /= 255.0 if a.max() > 1.5 else 1.0
    if shape_hw is not None and a.shape != tuple(shape_hw):
        a = cv2.resize(a, (shape_hw[1], shape_hw[0]), interpolation=cv2.INTER_LINEAR)
    return a


def zscore_ensemble(maps_by_model: dict, valid=None):
    """PHerc. 343 rule: per model, one mean/sd over ALL its depth maps (pooled), z-score each map, then average the
    models' z-scores at each depth. maps_by_model = {model: {depth: 2-D array}} (all on one grid).
    Returns {depth: ensemble z-map}."""
    stats = {}
    for m, by_d in maps_by_model.items():
        pooled = np.concatenate([(v[valid] if valid is not None else v.ravel()) for v in by_d.values()])
        stats[m] = (float(pooled.mean()), float(pooled.std() + 1e-6))
    depths = sorted(set.intersection(*[set(v) for v in maps_by_model.values()]))
    return {d: np.mean([(maps_by_model[m][d] - stats[m][0]) / stats[m][1] for m in maps_by_model], 0) for d in depths}


def product_rule(a, b):
    """PHerc. 306B rule: per-pixel product of two models' ink probabilities (ink only where both agree)."""
    return np.asarray(a, np.float32) * np.asarray(b, np.float32)


def claim_image(ink, ct_layer, out_png: str, voxel_um: float, threshold_pct: float = 85.0, full_pct: float = 99.5,
                ct_dim: float = 0.45, flip_lr: bool = False, flip_ud: bool = False, title: str | None = None):
    """Ink in yellow over a dimmed CT layer with a 1 cm scale bar (the PHerc. 343 look; percentiles instead of his
    fixed z-score levels). Returns pixels per cm."""
    import cv2
    ink = np.asarray(ink, np.float32)
    ct = np.asarray(ct_layer, np.float32)
    valid = ct > 0
    lo, hi = np.percentile(ink[valid], [threshold_pct, full_pct]) if valid.any() else (0, 1)
    op = np.clip((ink - lo) / max(hi - lo, 1e-6), 0, 1) * 0.9
    g = stretch(ct, 1, 99.5, valid).astype(np.float32) * ct_dim
    col = np.array([120, 235, 255], np.float32)          # BGR of (255, 235, 120)
    rgb = g[..., None] * (1 - op[..., None]) + col * op[..., None]
    rgb[~valid] = 0
    rgb = rgb.astype(np.uint8)
    if flip_lr:
        rgb = rgb[:, ::-1]
    if flip_ud:
        rgb = rgb[::-1]
    rgb = np.ascontiguousarray(rgb)
    px_cm = 10000.0 / voxel_um
    h = rgb.shape[0]
    y = h - 40
    x0 = 40
    cv2.rectangle(rgb, (x0, y - 12), (x0 + int(round(px_cm)), y), (255, 255, 255), -1)
    cv2.putText(rgb, "1 cm", (x0, y - 20), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)
    if title:
        cv2.putText(rgb, title, (40, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)
    cv2.imwrite(out_png, rgb)
    return px_cm


def depth_shuffle(stack, seed: int = 0):
    """False-positive control used by PHerc. 0306B: shuffle the layer order; letters must disappear."""
    rng = np.random.default_rng(seed)
    idx = rng.permutation(stack.shape[0])
    return np.ascontiguousarray(np.asarray(stack)[idx])

In [ ]:
%%writefile /content/scrolls19.json
{
 "PHerc0125": {
  "um": 9.362,
  "volume": "PHerc0125/volumes/20250821151825-9.362um-1.2m-113keV-masked.zarr",
  "shape_zyx": [
   20840,
   8387,
   8387
  ],
  "m7": "PHerc0125/representations/predictions/surfaces/20250821151825-surface-20260413222639-surface-m7-L0-th0.2.zarr",
  "normal_grids": "PHerc0125/representations/predictions/surfaces/20250821151825-surface-20260413222639-surface-m7-L0-th0.2.normal-grids",
  "lasagna": "PHerc0125/representations/predictions/lasagna/20250821151825-lasagna-20260419180421",
  "umbilicus": [
   "20250821151825-umbilicus-20260808111524.json"
  ],
  "sqm_median": 0.12,
  "sqm_best_band_z": 18432,
  "sqm_targets_xyz": [
   [
    5040,
    2736,
    18864
   ],
   [
    6576,
    5040,
    18096
   ],
   [
    5808,
    3504,
    17712
   ]
  ],
  "sqm_target_q": [
   0.79,
   0.75,
   0.68
  ]
 },
 "PHerc0175A": {
  "um": 8.64,
  "volume": "PHerc0175A/volumes/20250521115057-8.640um-1.2m-116keV-masked.zarr",
  "shape_zyx": [
   12748,
   9363,
   9363
  ],
  "m7": "PHerc0175A/representations/predictions/surfaces/20250521115057-surface-20260413222639-surface-m7-L0-th0.2.zarr",
  "normal_grids": "PHerc0175A/representations/predictions/surfaces/20250521115057-surface-20260413222639-surface-m7-L0-th0.2.normal-grids",
  "lasagna": null,
  "umbilicus": [],
  "sqm_median": 0.5,
  "sqm_best_band_z": 7680,
  "sqm_targets_xyz": [
   [
    5424,
    5808,
    6960
   ],
   [
    3120,
    5808,
    7728
   ],
   [
    4272,
    4656,
    7728
   ]
  ],
  "sqm_target_q": [
   0.93,
   0.93,
   0.93
  ]
 },
 "PHerc0175B": {
  "um": 8.64,
  "volume": "PHerc0175B/volumes/20250521125822-8.640um-1.2m-116keV-masked.zarr",
  "shape_zyx": [
   15897,
   9613,
   9613
  ],
  "m7": "PHerc0175B/representations/predictions/surfaces/20250521125822-surface-20260413222639-surface-m7-L0-th0.2.zarr",
  "normal_grids": "PHerc0175B/representations/predictions/surfaces/20250521125822-surface-20260413222639-surface-m7-L0-th0.2.normal-grids",
  "lasagna": null,
  "umbilicus": [],
  "sqm_median": 0.27,
  "sqm_best_band_z": 14592,
  "sqm_targets_xyz": [
   [
    5040,
    4656,
    13872
   ],
   [
    2352,
    2352,
    14256
   ],
   [
    1968,
    3888,
    14256
   ]
  ],
  "sqm_target_q": [
   0.84,
   0.8,
   0.7
  ]
 },
 "PHerc0191": {
  "um": 9.362,
  "volume": "PHerc0191/volumes/20250821151635-9.362um-1.2m-113keV-masked.zarr",
  "shape_zyx": [
   18977,
   8387,
   8387
  ],
  "m7": "PHerc0191/representations/predictions/surfaces/20250821151635-surface-20260413222639-surface-m7-L0-th0.2.zarr",
  "normal_grids": "PHerc0191/representations/predictions/surfaces/20250821151635-surface-20260413222639-surface-m7-L0-th0.2.normal-grids",
  "lasagna": "PHerc0191/representations/predictions/lasagna/20250821151635-lasagna-20260419180421",
  "umbilicus": [],
  "sqm_median": 0.41,
  "sqm_best_band_z": 11520,
  "sqm_targets_xyz": [
   [
    4656,
    4656,
    11952
   ],
   [
    2736,
    2736,
    11184
   ],
   [
    5424,
    5424,
    10800
   ]
  ],
  "sqm_target_q": [
   0.91,
   0.87,
   0.86
  ]
 },
 "PHerc0257": {
  "um": 9.362,
  "volume": "PHerc0257/volumes/20250821151750-9.362um-1.2m-113keV-masked.zarr",
  "shape_zyx": [
   18872,
   8388,
   8388
  ],
  "m7": "PHerc0257/representations/predictions/surfaces/20250821151750-surface-20260413222639-surface-m7-L0-th0.2.zarr",
  "normal_grids": "PHerc0257/representations/predictions/surfaces/20250821151750-surface-20260413222639-surface-m7-L0-th0.2.normal-grids",
  "lasagna": "PHerc0257/representations/predictions/lasagna/20250821151750-lasagna-20260419180421",
  "umbilicus": [],
  "sqm_median": 0.14,
  "sqm_best_band_z": 13056,
  "sqm_targets_xyz": [
   [
    5040,
    5808,
    13488
   ],
   [
    4272,
    4272,
    12720
   ],
   [
    4656,
    2736,
    12720
   ]
  ],
  "sqm_target_q": [
   0.79,
   0.78,
   0.72
  ]
 },
 "PHerc0268": {
  "um": 8.64,
  "volume": "PHerc0268/volumes/20251110183117-8.640um-1.2m-116keV-masked.zarr",
  "shape_zyx": [
   14833,
   12145,
   12145
  ],
  "m7": "PHerc0268/representations/predictions/surfaces/20251110183117-surface-20260413222639-surface-m7-L0-th0.2.zarr",
  "normal_grids": "PHerc0268/representations/predictions/surfaces/20251110183117-surface-20260413222639-surface-m7-L0-th0.2.normal-grids",
  "lasagna": "PHerc0268/representations/predictions/lasagna/20251110183117-lasagna-20260419180421",
  "umbilicus": [],
  "sqm_median": 0.08,
  "sqm_best_band_z": 13056,
  "sqm_targets_xyz": [
   [
    4272,
    10032,
    12720
   ],
   [
    2352,
    9264,
    12336
   ],
   [
    3504,
    8496,
    13104
   ]
  ],
  "sqm_target_q": [
   0.79,
   0.73,
   0.59
  ]
 },
 "PHerc0358": {
  "um": 9.362,
  "volume": "PHerc0358/volumes/20250821151737-9.362um-1.2m-113keV-masked.zarr",
  "shape_zyx": [
   14744,
   7783,
   7783
  ],
  "m7": "PHerc0358/representations/predictions/surfaces/20250821151737-surface-20260413222639-surface-m7-L0-th0.2.zarr",
  "normal_grids": "PHerc0358/representations/predictions/surfaces/20250821151737-surface-20260413222639-surface-m7-L0-th0.2.normal-grids",
  "lasagna": "PHerc0358/representations/predictions/lasagna/20250821151737-lasagna-20260419180421",
  "umbilicus": [
   "20250821151737-umbilicus-20260922235631.json"
  ],
  "sqm_median": 0.57,
  "sqm_best_band_z": 9216,
  "sqm_targets_xyz": [
   [
    3504,
    3504,
    9648
   ],
   [
    4272,
    4272,
    8496
   ],
   [
    1584,
    2352,
    10032
   ]
  ],
  "sqm_target_q": [
   0.95,
   0.95,
   0.93
  ]
 },
 "PHerc0483A": {
  "um": 8.64,
  "volume": "PHerc0483A/volumes/20250521140913-8.640um-1.2m-116keV-masked.zarr",
  "shape_zyx": [
   15898,
   8849,
   8849
  ],
  "m7": "PHerc0483A/representations/predictions/surfaces/20250521140913-surface-20260413222639-surface-m7-L0-th0.2.zarr",
  "normal_grids": "PHerc0483A/representations/predictions/surfaces/20250521140913-surface-20260413222639-surface-m7-L0-th0.2.normal-grids",
  "lasagna": null,
  "umbilicus": [],
  "sqm_median": 0.18,
  "sqm_best_band_z": 13056,
  "sqm_targets_xyz": [
   [
    3120,
    5808,
    13488
   ],
   [
    3888,
    4656,
    12720
   ],
   [
    6576,
    4272,
    12720
   ]
  ],
  "sqm_target_q": [
   0.85,
   0.85,
   0.83
  ]
 },
 "PHerc0483B": {
  "um": 8.64,
  "volume": "PHerc0483B/volumes/20251124083638-8.640um-1.2m-116keV-masked.zarr",
  "shape_zyx": [
   11880,
   8343,
   8343
  ],
  "m7": "PHerc0483B/representations/predictions/surfaces/20251124083638-surface-20260413222639-surface-m7-L0-th0.2.zarr",
  "normal_grids": "PHerc0483B/representations/predictions/surfaces/20251124083638-surface-20260413222639-surface-m7-L0-th0.2.normal-grids",
  "lasagna": null,
  "umbilicus": [],
  "sqm_median": 0.3,
  "sqm_best_band_z": 9984,
  "sqm_targets_xyz": [
   [
    4272,
    4656,
    10800
   ],
   [
    5808,
    3888,
    10800
   ],
   [
    1968,
    5424,
    9264
   ]
  ],
  "sqm_target_q": [
   0.89,
   0.87,
   0.81
  ]
 },
 "PHerc0490A": {
  "um": 8.64,
  "volume": "PHerc0490A/volumes/20250521151210-8.640um-1.2m-116keV-masked.zarr",
  "shape_zyx": [
   11698,
   9101,
   9101
  ],
  "m7": "PHerc0490A/representations/predictions/surfaces/20250521151210-surface-20260413222639-surface-m7-L0-th0.2.zarr",
  "normal_grids": "PHerc0490A/representations/predictions/surfaces/20250521151210-surface-20260413222639-surface-m7-L0-th0.2.normal-grids",
  "lasagna": null,
  "umbilicus": [],
  "sqm_median": 0.16,
  "sqm_best_band_z": 5376,
  "sqm_targets_xyz": [
   [
    2352,
    4272,
    6192
   ],
   [
    3504,
    3120,
    6192
   ],
   [
    5808,
    3120,
    5040
   ]
  ],
  "sqm_target_q": [
   0.94,
   0.9,
   0.87
  ]
 },
 "PHerc0490B": {
  "um": 8.64,
  "volume": "PHerc0490B/volumes/20250521151215-8.640um-1.2m-116keV-masked.zarr",
  "shape_zyx": [
   10648,
   8343,
   8343
  ],
  "m7": "PHerc0490B/representations/predictions/surfaces/20250521151215-surface-20260413222639-surface-m7-L0-th0.2.zarr",
  "normal_grids": "PHerc0490B/representations/predictions/surfaces/20250521151215-surface-20260413222639-surface-m7-L0-th0.2.normal-grids",
  "lasagna": null,
  "umbilicus": [],
  "sqm_median": 0.11,
  "sqm_best_band_z": 2304,
  "sqm_targets_xyz": [
   [
    3888,
    3504,
    2352
   ],
   [
    6192,
    2736,
    2352
   ],
   [
    5040,
    5040,
    1968
   ]
  ],
  "sqm_target_q": [
   0.83,
   0.68,
   0.63
  ]
 },
 "PHerc0800": {
  "um": 8.64,
  "volume": "PHerc0800/volumes/20250521135224-8.640um-1.2m-116keV-masked.zarr",
  "shape_zyx": [
   24298,
   9867,
   9867
  ],
  "m7": "PHerc0800/representations/predictions/surfaces/20250521135224-surface-20260413222639-surface-m7-L0-th0.2.zarr",
  "normal_grids": "PHerc0800/representations/predictions/surfaces/20250521135224-surface-20260413222639-surface-m7-L0-th0.2.normal-grids",
  "lasagna": "PHerc0800/representations/predictions/lasagna/20250521135224-lasagna-20260419180421",
  "umbilicus": [
   "20250521135224-umbilicus-20260924033751.json"
  ],
  "sqm_median": 0.28,
  "sqm_best_band_z": 3072,
  "sqm_targets_xyz": [
   [
    3504,
    3120,
    2736
   ],
   [
    6960,
    5808,
    3504
   ],
   [
    2736,
    3888,
    3888
   ]
  ],
  "sqm_target_q": [
   0.89,
   0.88,
   0.88
  ]
 },
 "PHerc0813": {
  "um": 9.362,
  "volume": "PHerc0813/volumes/20250821151723-9.362um-1.2m-113keV-masked.zarr",
  "shape_zyx": [
   16993,
   7947,
   7947
  ],
  "m7": "PHerc0813/representations/predictions/surfaces/20250821151723-surface-20260413222639-surface-m7-L0-th0.2.zarr",
  "normal_grids": "PHerc0813/representations/predictions/surfaces/20250821151723-surface-20260413222639-surface-m7-L0-th0.2.normal-grids",
  "lasagna": "PHerc0813/representations/predictions/lasagna/20250821151723-lasagna-20260419180421",
  "umbilicus": [
   "20250821151723-umbilicus-20260924033203.json"
  ],
  "sqm_median": 0.53,
  "sqm_best_band_z": 12288,
  "sqm_targets_xyz": [
   [
    3504,
    1968,
    13104
   ],
   [
    2736,
    3504,
    13104
   ],
   [
    5424,
    4656,
    12336
   ]
  ],
  "sqm_target_q": [
   0.95,
   0.94,
   0.92
  ]
 },
 "PHerc0826": {
  "um": 9.362,
  "volume": "PHerc0826/volumes/20250821151701-9.362um-1.2m-113keV-masked.zarr",
  "shape_zyx": [
   16920,
   8169,
   8169
  ],
  "m7": "PHerc0826/representations/predictions/surfaces/20250821151701-surface-20260413222639-surface-m7-L0-th0.2.zarr",
  "normal_grids": "PHerc0826/representations/predictions/surfaces/20250821151701-surface-20260413222639-surface-m7-L0-th0.2.normal-grids",
  "lasagna": "PHerc0826/representations/predictions/lasagna/20250821151701-lasagna-20260419180421",
  "umbilicus": [
   "20250821151701-umbilicus-20260808113303.json"
  ],
  "sqm_median": 0.44,
  "sqm_best_band_z": 9216,
  "sqm_targets_xyz": [
   [
    5808,
    4656,
    8496
   ],
   [
    1584,
    4656,
    10032
   ],
   [
    4656,
    5808,
    9648
   ]
  ],
  "sqm_target_q": [
   0.94,
   0.94,
   0.93
  ]
 },
 "PHerc0846A": {
  "um": 9.362,
  "volume": "PHerc0846A/volumes/20250728152254-9.362um-1.2m-113keV-masked.zarr",
  "shape_zyx": [
   14019,
   7726,
   7726
  ],
  "m7": "PHerc0846A/representations/predictions/surfaces/20250728152254-surface-20260413222639-surface-m7-L0-th0.2.zarr",
  "normal_grids": "PHerc0846A/representations/predictions/surfaces/20250728152254-surface-20260413222639-surface-m7-L0-th0.2.normal-grids",
  "lasagna": null,
  "umbilicus": [],
  "sqm_median": 0.18,
  "sqm_best_band_z": 3072,
  "sqm_targets_xyz": [
   [
    5808,
    3504,
    3504
   ],
   [
    2736,
    2352,
    3888
   ],
   [
    4272,
    2352,
    3888
   ]
  ],
  "sqm_target_q": [
   0.91,
   0.89,
   0.87
  ]
 },
 "PHerc0846B": {
  "um": 9.362,
  "volume": "PHerc0846B/volumes/20250804142305-9.362um-1.2m-113keV-masked.zarr",
  "shape_zyx": [
   13926,
   7859,
   7859
  ],
  "m7": "PHerc0846B/representations/predictions/surfaces/20250804142305-surface-20260413222639-surface-m7-L0-th0.2.zarr",
  "normal_grids": "PHerc0846B/representations/predictions/surfaces/20250804142305-surface-20260413222639-surface-m7-L0-th0.2.normal-grids",
  "lasagna": null,
  "umbilicus": [],
  "sqm_median": 0.25,
  "sqm_best_band_z": 11520,
  "sqm_targets_xyz": [
   [
    5808,
    3504,
    11952
   ],
   [
    3504,
    4272,
    11184
   ],
   [
    1200,
    3504,
    12336
   ]
  ],
  "sqm_target_q": [
   0.88,
   0.87,
   0.87
  ]
 },
 "PHerc1203": {
  "um": 9.362,
  "volume": "PHerc1203/volumes/20250820131727-9.362um-1.2m-113keV-masked.zarr",
  "shape_zyx": [
   18977,
   6844,
   6844
  ],
  "m7": "PHerc1203/representations/predictions/surfaces/20250820131727-surface-20260413222639-surface-m7-L0-th0.2.zarr",
  "normal_grids": "PHerc1203/representations/predictions/surfaces/20250820131727-surface-20260413222639-surface-m7-L0-th0.2.normal-grids",
  "lasagna": "PHerc1203/representations/predictions/lasagna/20250820131727-lasagna-20260419180421",
  "umbilicus": [
   "20250820131727-umbilicus-20260924035533.json"
  ],
  "sqm_median": 0.35,
  "sqm_best_band_z": 11520,
  "sqm_targets_xyz": [
   [
    2736,
    3120,
    12720
   ],
   [
    5040,
    3888,
    12336
   ],
   [
    4272,
    2352,
    12720
   ]
  ],
  "sqm_target_q": [
   0.92,
   0.9,
   0.88
  ]
 },
 "PHerc1218": {
  "um": 8.64,
  "volume": "PHerc1218/volumes/20250521120456-8.640um-1.2m-116keV-masked.zarr",
  "shape_zyx": [
   23247,
   7593,
   7593
  ],
  "m7": "PHerc1218/representations/predictions/surfaces/20250521120456-surface-20260413222639-surface-m7-L0-th0.2.zarr",
  "normal_grids": "PHerc1218/representations/predictions/surfaces/20250521120456-surface-20260413222639-surface-m7-L0-th0.2.normal-grids",
  "lasagna": "PHerc1218/representations/predictions/lasagna/20250521120456-lasagna-20260419180421",
  "umbilicus": [],
  "sqm_median": 0.17,
  "sqm_best_band_z": 7680,
  "sqm_targets_xyz": [
   [
    3504,
    3888,
    7344
   ],
   [
    2352,
    3504,
    8496
   ],
   [
    5040,
    3504,
    7728
   ]
  ],
  "sqm_target_q": [
   0.92,
   0.81,
   0.79
  ]
 },
 "PHerc1545": {
  "um": 9.362,
  "volume": "PHerc1545/volumes/20250821151648-9.362um-1.2m-113keV-masked.zarr",
  "shape_zyx": [
   20961,
   7506,
   7506
  ],
  "m7": "PHerc1545/representations/predictions/surfaces/20250821151648-surface-20260413222639-surface-m7-L0-th0.2.zarr",
  "normal_grids": "PHerc1545/representations/predictions/surfaces/20250821151648-surface-20260413222639-surface-m7-L0-th0.2.normal-grids",
  "lasagna": "PHerc1545/representations/predictions/lasagna/20250821151648-lasagna-20260419180421",
  "umbilicus": [],
  "sqm_median": 0.31,
  "sqm_best_band_z": 15360,
  "sqm_targets_xyz": [
   [
    3888,
    4656,
    15408
   ],
   [
    1968,
    3504,
    16176
   ],
   [
    5424,
    5424,
    14640
   ]
  ],
  "sqm_target_q": [
   0.87,
   0.86,
   0.82
  ]
 }
}

In [ ]:
sys.path.insert(0, "/content")
import importlib, numpy as np
import fl_helpers as H; importlib.reload(H)
SCROLLS = json.load(open("/content/scrolls19.json"))
S = SCROLLS[SCROLL]; UM = S["um"]
print(json.dumps({k: S[k] for k in ("um", "shape_zyx", "sqm_median", "sqm_best_band_z", "sqm_targets_xyz", "sqm_target_q")}, indent=1))
print("Lasagna predictions:", bool(S["lasagna"]), "| umbilicus:", S["umbilicus"] or "none published")
VOL_URL, M7_URL = H.url(S["volume"]), H.url(S["m7"])

## 4. Install: system libraries and the VC3D AppImage (headless CLI tools) — `RAN ON COLAB T4` (job rcp_001: extract + `list` + `--help` worked; 19 tools)
The Linux AppImage of VC3D bundles the GUI **and** ~35 command-line tools, including `vc_grow_seg_from_seed` (GrowPatch) and `vc_render_tifxyz`. It runs headless: the PHerc. 343 winner rendered with it (`QT_QPA_PLATFORM=offscreen`). Colab has no FUSE, so we extract it once. The host must provide the OpenGL/font libraries (list from villa's `vc3d-linux.yml` smoke test and the 343 Dockerfile).

The `latest` release is replaced with every build of villa `main`. If the download below fails (404), open https://github.com/ScrollPrize/villa/releases/tag/latest, copy the current `VC3D-…-linux-x86_64.AppImage` link into `APPIMAGE_URL` and set `APPIMAGE_SHA256 = None`.

In [ ]:
APPIMAGE_URL = "https://github.com/ScrollPrize/villa/releases/download/latest/VC3D-c8f3ba4-2026-11-20-linux-x86_64.AppImage"
APPIMAGE_SHA256 = "ed4846d5985ac3e67e2a5881e8c915a8fab111289a021f896cb49ab96dc290f9"   # measured 2026-11-20
VC = Path("/content/vc3d"); VC.mkdir(exist_ok=True)
sh = lambda c: subprocess.run(c, shell=True, check=True)
sh("apt-get -qq update && apt-get -qq install -y --no-install-recommends libglvnd0 libglx0 libegl1 libopengl0 libgl1 "
   "libgl1-mesa-dri libglx-mesa0 libx11-6 libx11-xcb1 libxcb1 libice6 libsm6 libfontconfig1 libfreetype6 libharfbuzz0b "
   "libxkbcommon0 libdbus-1-3 > /dev/null")
if not (VC / "squashfs-root/AppRun").exists():
    sh(f"curl -fL -o {VC}/VC3D.AppImage '{APPIMAGE_URL}'")
    if APPIMAGE_SHA256:
        got = subprocess.run(f"sha256sum {VC}/VC3D.AppImage", shell=True, capture_output=True, text=True).stdout.split()[0]
        assert got == APPIMAGE_SHA256, f"AppImage changed: {got} (set APPIMAGE_SHA256=None to accept a newer build)"
    sh(f"chmod +x {VC}/VC3D.AppImage && cd {VC} && ./VC3D.AppImage --appimage-extract > /dev/null")
APPRUN = str(VC / "squashfs-root/AppRun")
# VC3D settings: where streamed CT / prediction chunks are cached, and a low free-space floor for Colab
CFG = Path("/content/vc3d_cfg"); CFG.mkdir(exist_ok=True)
(CFG / "VC3D.ini").write_text("[viewer]\nremote_cache_dir=/content/vc3d_cache\n[perf]\nremote_cache_min_free_gib=5\n")
VC_ENV = dict(os.environ, QT_QPA_PLATFORM="offscreen", VC3D_CONFIG_DIR=str(CFG), OMP_NUM_THREADS=str(os.cpu_count()))
def vc(tool, *args, log=None):
    cmd = [APPRUN, tool, *map(str, args)]
    print("$", " ".join(cmd))
    with open(log or os.devnull, "w") as f:
        r = subprocess.run(cmd, env=VC_ENV, stdout=f if log else None, stderr=subprocess.STDOUT if log else None)
    if r.returncode != 0:
        if log: print(open(log).read()[-3000:])
        raise RuntimeError(f"{tool} failed ({r.returncode})")
tools = subprocess.run([APPRUN, "list"], env=VC_ENV, capture_output=True, text=True).stdout.split()
print(len(tools), "tools;", [t for t in tools if t in ("vc_grow_seg_from_seed", "vc_render_tifxyz", "vc_tifxyz2obj")])
assert "vc_grow_seg_from_seed" in tools and "vc_render_tifxyz" in tools
tick("appimage")

## 5. Ink models — `SOURCE-VERIFIED, UNTESTED`
Hecate 9.6 µm (`scrollprize/hecate`) is a single script plus weights and runs on Colab's own PyTorch. `hecate.py`'s SHA-256 is the one pinned by the PHerc. 343 winner (`ink/models.tsv`). The ink_9um and Reader V3 checkpoints are only used if `RUN_VILLA_INK = True`.

In [ ]:
from huggingface_hub import hf_hub_download
MD = Path("/content/models"); MD.mkdir(exist_ok=True)
HEC_PY  = hf_hub_download("scrollprize/hecate", "hecate.py", local_dir=MD / "hecate")
HEC_PTH = hf_hub_download("scrollprize/hecate", "hecate_9.6um.pth", local_dir=MD / "hecate")
assert subprocess.run(f"sha256sum {HEC_PY}", shell=True, capture_output=True, text=True).stdout.startswith(
    "c232c18a1a86cfb91257a00db13288202bb8ae33732ba26f7291a9b8adb8da59"), "hecate.py changed upstream: read it before running"
if RUN_VILLA_INK:
    CK42 = hf_hub_download("scrollprize/ink_9um", "hybrid_3d2d-seed42/step-075000.pth", local_dir=MD / "ink_9um")
    CK43 = hf_hub_download("scrollprize/ink_9um", "hybrid_3d2d-seed43/step-075000.pth", local_dir=MD / "ink_9um")
    CKR3 = hf_hub_download("domenicor046/reader-v3", "reader-v3.pth", local_dir=MD / "reader-v3")
import torch
GPU = torch.cuda.is_available()
BF16 = GPU and torch.cuda.is_bf16_supported() and "T4" not in torch.cuda.get_device_name(0)
print("torch", torch.__version__, "| GPU", torch.cuda.get_device_name(0) if GPU else None, "| bf16", BF16)
tick("models")

### Hecate runner
`hecate.py` needs a uint8 (Z, Y, X) stack at 9.6 µm in **all three axes** (it refuses spacings more than 2 % off, so both 8.64 and 9.362 µm renders are resampled), and uses the **central 16 planes** of whatever it is given. We therefore resample the render once and hand it exactly 16 planes centred on surface + d. `--reverse` flips the stack, i.e. reads the other face; we run both at d = 0 and keep the face that shows structure. The output map is resized back to the native render grid.

In [ ]:
import cv2
def run_hecate(res, n_native, d, out_png, reverse=False):
    if os.path.exists(out_png): return out_png
    crop = H.hecate_crop(res, n_native, UM, d)
    tmp = out_png + ".in.npy"; np.save(tmp, crop)
    cmd = [sys.executable, HEC_PY, "--checkpoint", HEC_PTH, "--input", tmp, "--spacing-um", "9.6", "--output", out_png,
           "--device", "cuda" if GPU else "cpu", "--batch-size", str(HECATE_BATCH), "--precision", "bf16" if BF16 else "fp32"]
    if reverse: cmd.append("--reverse")
    t = time.time(); subprocess.run(cmd, check=True, cwd="/content"); os.remove(tmp)
    print(f"hecate d={d:+d} {'rev' if reverse else 'fwd'}: {time.time()-t:.0f}s -> {out_png}")
    return out_png

## 6. (first session) Positive control: PHerc. 343's winning segment — `SOURCE-VERIFIED, UNTESTED`
Before spending hours on an unread scroll, check that this exact pipeline shows the known letters. We download the 343 winner's published mesh (`outputs/concat_w047-w048_R5B2_z9500-11000.tifxyz`, repo commit `e8370f1`), render it the same way, run Hecate at d = +1 (the depth of his image) in both directions, and write `control_343_*.png` to Drive. The letters πᾶν τὸ γινόμε… should be visible in one direction. If they are not, fix the pipeline before going on. About 20-40 min on a T4.

In [ ]:
if RUN_POSITIVE_CONTROL:
    C = Path(WORK) / "control343"; M = C / "mesh.tifxyz"; M.mkdir(parents=True, exist_ok=True)
    raw = "https://raw.githubusercontent.com/Nieuwlaar/pherc343-first-letters/e8370f1f88a211166056df62ea6fcfffe4a1063e/outputs/concat_w047-w048_R5B2_z9500-11000.tifxyz/"
    for f in ("x.tif", "y.tif", "z.tif", "meta.json"):
        if not (M / f).exists(): sh(f"curl -fsL -o {M/f} {raw}{f}")
    vol343 = "https://vesuvius-challenge-open-data.s3.us-east-1.amazonaws.com/PHerc0343/volumes/20250521140437-8.640um-1.2m-116keV-masked.zarr"
    R343 = C / "render41.zarr"
    if not (R343 / "0").exists():
        vc("vc_render_tifxyz", "--volume", vol343, "--group-idx", 0, "--scale", 1, "--segmentation", M,
           "--num-slices", 41, "--slice-step", 1, "--cache-gb", 8, "--zarr-output", R343, log=str(C / "render.log"))
    r343 = H.open_render(str(R343)); print("control render", r343.shape)
    # his letter rows lie in raw render rows 105..2560, cols 380..6021 (outputs/*.json "area"); use that crop
    UM_SAVE, UM = UM, 8.64
    res343 = H.resample_stack(np.asarray(r343[:, 105:2560, 380:6021]), 8.64, 9.6, device="cuda" if GPU else "cpu")
    for rev in (False, True):
        p = run_hecate(res343, 41, +1, str(C / f"hecate_d+1_{'rev' if rev else 'fwd'}.png"), reverse=rev)
        ink = H.load_map(p, shape_hw=(2455, 5641))
        H.claim_image(ink, r343[21, 105:2560, 380:6021], str(C / f"control_343_{'rev' if rev else 'fwd'}.png"), 8.64,
                      flip_lr=True, flip_ud=True, title=f"PHerc0343 control, Hecate d=+1 {'reverse' if rev else 'forward'}")
    UM = UM_SAVE
    for f in C.glob("control_343_*.png"): shutil.copy(f, DRIVE_DIR)
    tick("positive control")

## 7. Choose the seed — `SMOKE-TESTED (CPU)`
How the winners chose *where*: 306B ran a spiral fit over the scroll's **clearest 1,500-slice band** of the Scan-Quality-Map atlas, inked 40 windings and ranked 72 windows by a text-row score and a small glyph classifier; 343 found "texture which was oddly well structured/sized" in a band of above-median scan quality ~7 mm from the scroll axis; 211's region was one of "remarkably regular" structure. Here we take the clearest SQM region inside your window and snap it onto the m7 sheet. Look at the CT numbers it prints: a seed in a gap or in haze will not grow well.

In [ ]:
if SEED_XYZ is None:
    cands = [(q, t) for t, q in zip(S["sqm_targets_xyz"], S["sqm_target_q"]) if Z0 <= t[2] < Z1]
    if not cands:
        print(f"no SQM target in z {Z0}-{Z1}; targets are {S['sqm_targets_xyz']} (band {S['sqm_best_band_z']}).",
              "Using the window centre on the scroll axis offset is NOT safe - set SEED_XYZ by hand (VC3D or the CT viewer).")
        raise SystemExit
    cands.sort(key=lambda c: -c[0]); q, t = cands[min(SQM_TARGET, len(cands) - 1)]
    print("SQM target", t, "quality", q); SEED_XYZ = tuple(t)
snap = H.snap_seed(S["m7"], SEED_XYZ, half=48)
assert snap, "no m7 sheet within 48 voxels: choose another point"
SEED = snap[:3]; print("seed on m7 sheet:", SEED, f"(moved {snap[3]:.1f} vox)")
print("CT around seed:", H.sheet_profile(S["volume"], SEED, half=32))

## 8. Normal grids for GrowPatch — `RAN ON COLAB T4` (job rcp_012: PHerc0175A, 9.4 GB in 21 min — the slowest step)
GrowPatch follows the organisers' m7 **normal grids** (one `.grid` file per xy / xz / yz slice). We copy the store to local disk (7-11 GB: PHerc0358 7.0 GB for a 1,000-slice window, PHerc0175A 10.6 GB for the growth range used here; xz and yz grids span all z). A streaming cache also exists in VC3D, but it requires the directory to sit at an exact place inside the remote cache, so a plain local copy is the safer headless route.

In [ ]:
NG = Path(WORK) / "normal_grids" / SCROLL
ZM = 2 * 20 * GENERATIONS + 200            # growth can reach ~20 voxels per generation in z, both ways
H.download_normal_grids(S["normal_grids"], str(NG), max(0, SEED[2] - ZM), SEED[2] + ZM, margin=0, threads=32)
tick("normal grids")

## 9. GrowPatch (headless `vc_grow_seg_from_seed`) — `RAN ON COLAB T4` (job rcp_012: 80 generations, 2 threads, 489 s → 162×162-node mesh, 7.45 cm²)
Parameters = the 306B winners' claim-surface run (`discovery/fl/sqm/PHerc0306B_f306_w074own/params.json`): seed mode, step 20, `min_area_cm` 0.5, `thread_limit` 4, `voxelsize` = scan voxel size, plus our `GENERATIONS`. CLI flags (`-v -t -p --seed`) from `volume-cartographer/apps/src/vc_grow_seg_from_seed.cpp`; `-v` is the m7 prediction URL (streamed). CPU only; 15-60 min depending on cores and generations.

In [ ]:
G = Path(WORK) / "grow" / RUN_TAG; G.mkdir(parents=True, exist_ok=True)
H.write_grow_params(str(G / "params.json"), str(NG), UM, generations=GENERATIONS, threads=max(2, os.cpu_count()))
if not glob.glob(str(G / "auto_grown_*")):
    vc("vc_grow_seg_from_seed", "-v", M7_URL, "-t", G, "-p", G / "params.json", "--seed", *SEED, log=str(G / "grow.log"))
MESH = sorted(glob.glob(str(G / "auto_grown_*")))[-1]
umb = None
if S["umbilicus"]:
    umb = str(G / "umbilicus.json"); sh(f"curl -fsL -o {umb} {H.url(SCROLL + '/representations/umbilicus/' + S['umbilicus'][0])}")
print(MESH); print(json.dumps(H.tifxyz_stats(MESH, umb, UM), indent=1))
tick("growpatch")

**Check the surface before inking (manual, important).** All three winners spent most of their time here. Open the mesh in VC3D on your own computer if you can (File → Attach Segments, streaming the same volume) and check in cross-section that it stays on one sheet; a patch that hops sheets hides or fakes ink. The 306B winners moved the drifting right-hand end of their patch back onto the sheet by hand; the 343 winner snapped every node to the nearest m7 sheet along its normal (`mesh/snap_meshes.py` in his repo).

## 10. Render 41 layers — `RAN ON COLAB T4` (job rcp_013: 3240×3240×41 in 250 s, 517 MB; 2.3 GB CT chunk cache)
`vc_render_tifxyz --volume <CT URL> --group-idx 0 --scale 1 --num-slices 41 --slice-step 1 --zarr-output …` streams only the chunks the surface touches. Layer k samples the CT at (k − 20) voxels along the mesh normal, so layer 20 is the surface. No `--flip-normals`, as in the 343 and 306B renders. (`--cuda` would sample on the GPU; left off because untested.)

In [ ]:
RND = Path(WORK) / "render" / f"{RUN_TAG}_L{N_LAYERS}.zarr"
if not (RND / "0").exists():
    vc("vc_render_tifxyz", "--volume", VOL_URL, "--group-idx", 0, "--scale", 1, "--segmentation", MESH,
       "--num-slices", N_LAYERS, "--slice-step", 1, "--cache-gb", 8, "--zarr-output", RND, log=str(RND) + ".log")
render = H.open_render(str(RND)); print("render", render.shape, render.dtype)
QL = Path(WORK) / "quick" / RUN_TAG
print(H.quicklook(render, str(QL)))
for f in QL.glob("*.png"): shutil.copy(f, f"{DRIVE_DIR}/{RUN_TAG}_{f.name}")
tick("render")

Look at `composite_max.png` and `layer_mid.png` on Drive now. You should see papyrus fibres (a woven grid). Large black areas = the patch is off the papyrus; a smeared band = haze. Directly visible ink (bright, cracked texture) counts even without a model.

## 11. Hecate depth sweep, both faces — `SOURCE-VERIFIED, UNTESTED` (resampling and crops `SMOKE-TESTED (CPU)`)

**Measured warning:** Hecate on a T4 ran at only ~11.6 cm²/h per pass (bf16 autocast is not accelerated on a T4; job rcp_024 timed out at 90 min on a 7.45 cm² patch). On a T4 use `precision fp32` (set below) and keep the sweep short, or run Hecate only on a small region; reader-v3 / ink_9um run at 350–625 cm²/h on the same GPU.
First d = 0 in both directions; then the full `DEPTHS` sweep on the face you pick. Hecate is depth-sharp (±4 layers costs ~0.1 AUC; averaging d = −1, 0, +1 helps — ArcheyChen/vesuvius-eligible-scouting).

In [ ]:
HD = Path(WORK) / "maps" / RUN_TAG; HD.mkdir(parents=True, exist_ok=True)
res = H.resample_stack(render, UM, 9.6, device="cuda" if GPU else "cpu"); print("resampled", res.shape)
HW = render.shape[1:]
for rev in (False, True):
    run_hecate(res, N_LAYERS, 0, str(HD / f"hecate_d+0_{'rev' if rev else 'fwd'}.png"), reverse=rev)
ct0 = np.asarray(render[H.surface_index(N_LAYERS)])
for rev in (False, True):
    m = H.load_map(str(HD / f"hecate_d+0_{'rev' if rev else 'fwd'}.png"), HW)
    p = str(HD / f"view_hecate_d+0_{'rev' if rev else 'fwd'}.png")
    H.claim_image(m, ct0, p, UM, title=f"{RUN_TAG} hecate d=0 {'reverse' if rev else 'forward'}"); shutil.copy(p, DRIVE_DIR)
tick("hecate d=0")

In [ ]:
# Pick the face: look at the two view_* images on Drive. Letters, if any, appear on ONE face only.
# Default guess: the face with the higher 99.5th percentile of ink probability.
score = {rev: np.percentile(H.load_map(str(HD / f"hecate_d+0_{'rev' if rev else 'fwd'}.png")), 99.5) for rev in (False, True)}
REVERSE = max(score, key=score.get)          # override by hand: REVERSE = True / False
print("p99.5 per face:", score, "-> using", "reverse" if REVERSE else "forward")
for d in DEPTHS:
    run_hecate(res, N_LAYERS, d, str(HD / f"hecate_d{d:+d}_{'rev' if REVERSE else 'fwd'}.png"), reverse=REVERSE)
tick("hecate sweep")

## 12. (optional) villa environment and ink_9um seeds 42 + 43, Reader V3 — `SOURCE-VERIFIED, UNTESTED`

**What actually ran on the T4 instead (job `rcp_025_ink_fast.sh`, recommended):** no villa build at all — the ink agent's `infer_ink.py` (vendored ink_9um/reader-v3 model code, plain PyTorch) with `--model ink9um -c reader-v3.pth` and then `-c seed42 seed43`, `--input-um 8.64 --surface-layer 20 --z-shifts=-3,-2,-1,0,1 --direction both --save-each --device cuda`: 7 + 16 min for 7.45 cm², <3 GB VRAM. If you have that script, use it in place of this section.
The official inference CLI (`python -m vesuvius.ink_detection.inference.infer`) imports villa's native `vc_reader`, so the villa packages must be **compiled** (Python 3.14, C++ with gcc ≥ 13). This is villa's own bootstrap (`lasagna/scripts/bootstrap_venv.py`, as in the 343 winner's Dockerfile) plus villa's apt dependency script. Expect 20-60 min and possibly a failure on Colab's older Ubuntu; if it fails, keep the Hecate results and move on. Flags checked in `infer.py` at `c8f3ba4`: `--layer-start/--layer-end` (end exclusive) select a 17-layer window centred on surface + d; `--ensemble-checkpoint` averages seed 43 with seed 42 inside one run; `--direction both` writes `out.tif` and `out_reverse.tif`. Reader V3's author says it is a drop-in checkpoint for the same CLI (on another villa branch); untested here.

In [ ]:
if RUN_VILLA_INK:
    V = Path("/content/villa")
    if not V.exists():
        sh("git clone --filter=blob:none https://github.com/ScrollPrize/villa.git /content/villa && cd /content/villa && git checkout -q c8f3ba45304f0a317a49a155929e30d294f201eb")
    sh("command -v uv || pip -q install uv")
    sh("bash /content/villa/volume-cartographer/scripts/install_build_deps.sh minimal-build > /content/apt_build.log 2>&1")
    if not shutil.which("g++-13"):   # Ubuntu 22.04 ships gcc 11
        sh("add-apt-repository -y ppa:ubuntu-toolchain-r/test > /dev/null && apt-get -qq update && apt-get -qq install -y g++-13 > /dev/null")
    sh("cd /content/villa/lasagna && python3 scripts/bootstrap_venv.py --venv /content/vvenv --backend cu128 > /content/bootstrap.log 2>&1 || (tail -40 /content/bootstrap.log; false)")
    VPY = "/content/vvenv/bin/python"
    tick("villa env")

In [ ]:
if RUN_VILLA_INK:
    def villa_ink(ck, d, tag, extra=()):
        s, e = H.villa_layer_window(N_LAYERS, d)
        out = HD / f"{tag}_d{d:+d}.tif"
        if (HD / f"{tag}_d{d:+d}_reverse.tif").exists(): return
        sh(f"cd /content && {VPY} -m vesuvius.ink_detection.inference.infer {RND} {ck} {out} --overlap 0.5 --blend-mode hann "
           f"--batch-size 4 --direction both --layer-start {s} --layer-end {e} {' '.join(extra)}")
    for d in DEPTHS:
        villa_ink(CK42, d, "ink9_42_43", ("--ensemble-checkpoint", CK43))
        villa_ink(CKR3, d, "readerv3")
    tick("villa ink")

## 13. Ensembles, images, and a depth-shuffle control — `SMOKE-TESTED (CPU)` on synthetic maps
* **343 rule**: each model's maps are z-scored with one mean/sd pooled over all its depths, then the models are averaged at each depth.
* **306B rule**: product of two models' probabilities (ink only where both agree); shown here as Hecate × ink_9um when both exist.
* **Control**: Hecate on the same render with its layers shuffled. Letters that survive shuffling are not ink.

Which villa output file is the "reading side"? With no `--flip-normals`, the 343 winner found `*_reverse.tif` was the reading side on his mesh; mesh orientation differs between tools, so check both. Here we pair villa's `_reverse` with Hecate `--reverse`, and plain with plain.

In [ ]:
side = "rev" if REVERSE else "fwd"
maps = {"hecate": {d: H.load_map(str(HD / f"hecate_d{d:+d}_{side}.png"), HW) for d in DEPTHS}}
if RUN_VILLA_INK:
    suf = "_reverse" if REVERSE else ""
    for tag in ("ink9_42_43", "readerv3"):
        ps = {d: HD / f"{tag}_d{d:+d}{suf}.tif" for d in DEPTHS}
        if all(p.exists() for p in ps.values()):
            maps[tag] = {d: H.load_map(str(p), HW) for d, p in ps.items()}
valid = np.asarray(render[H.surface_index(N_LAYERS)]) > 0
ens = H.zscore_ensemble(maps, valid)
OUT = Path(DRIVE_DIR) / RUN_TAG; OUT.mkdir(exist_ok=True)
for d in DEPTHS:
    ct = np.asarray(render[H.surface_index(N_LAYERS) + d])
    H.claim_image(ens[d], ct, str(OUT / f"ensemble_z_d{d:+d}.png"), UM, title=f"{RUN_TAG} z-score ensemble {list(maps)} d={d:+d}")
dsum = np.mean([ens[d] for d in DEPTHS if abs(d) <= 1], 0)
H.claim_image(dsum, ct0, str(OUT / "ensemble_z_d-1..+1.png"), UM, title=f"{RUN_TAG} ensemble mean of d=-1..+1")
if "ink9_42_43" in maps:
    H.claim_image(H.product_rule(maps["hecate"][0], maps["ink9_42_43"][0]), ct0, str(OUT / "product_hecate_x_ink9_d+0.png"), UM,
                  threshold_pct=50, full_pct=99.9, title=f"{RUN_TAG} product rule d=0")
# depth-shuffle control (Hecate, d=0)
shuf = H.resample_stack(H.depth_shuffle(render), UM, 9.6, device="cuda" if GPU else "cpu")
pc = run_hecate(shuf, N_LAYERS, 0, str(HD / f"CONTROL_shuffled_hecate_d+0_{side}.png"), reverse=REVERSE)
H.claim_image(H.load_map(pc, HW), ct0, str(OUT / "CONTROL_depth_shuffled.png"), UM, title="CONTROL: layers shuffled - must show no letters")
json.dump({"scroll": SCROLL, "z_window": [Z0, Z1], "seed_xyz": SEED, "mesh": MESH, "render": str(RND), "layers": N_LAYERS,
           "depths": DEPTHS, "face": side, "models": list(maps), "minutes": TIMES}, open(OUT / "run.json", "w"), indent=1)
tick("images"); print("images in", OUT)

## 14. What to do with the result
* **Judge by shape, not by area.** Fibre-grid texture produces "ink" over large areas; letters are 2-4 mm tall (230-460 px at 8.64 µm), strokes 0.3-0.5 mm, and sit in rows parallel to the horizontal fibres. Compare with `control_343_*.png`.
* **Something letter-like?** Re-run with a wider `DEPTHS` (343 used −15…+15), grow a second patch from a neighbouring seed on the same sheet, and check the surface in VC3D. Keep it private.
* **Nothing?** That is the usual outcome. Move the seed along the same sheet or to the next SQM target, then to another scroll. A clean negative with images, commands and timings is publishable and has earned progress prizes (PHerc. 0826, $1,000).
* **Budget:** see `run.json` → `minutes`. GrowPatch and rendering are CPU-bound: on Colab Pro, consider running sections 7-10 on a CPU runtime (cheaper), copying `fl_work` to Drive, then switching to a GPU runtime for 11-13.

## Appendix A — the spiral-fitter route (PHerc. 343, 211; 306B's screening) — `UNTESTED`
Only for scrolls with Lasagna predictions (`S["lasagna"]` above: 0125, 0191, 0257, 0268, 0358, 0800, 0813, 0826, 1203, 1218, 1545). At villa `c8f3ba4` the command-line fitter takes an input manifest and an object store; the older `--dataset` + `spiral-scroll.json` interface the winners used (villa `0fb38c45`) is refused. Steps, each checked against `spiral-fitting/README.md` and the source, never run:
1. Build the villa env as in section 12 and `cd /content/villa/spiral-fitting && uv sync` (builds `vc_spiral` native helpers; Python ≥ 3.14).
2. Tracks: download `https://dl.ash2txt.org/datasets/spiral_datasets/<SCROLL>/<volume id>/tracks/<SCROLL>_<volume id>_surface_m7_L0_th0.2.dbm.vctracks/` (9 files; this is how the 306B team fetched them).
3. Object store in Python (the `vc_object_ingest` CLI is not in the AppImage): `from vc_object_store import ObjectStore; with ObjectStore.create("scroll.vcobj", author="me") as st: st.ingest_geometry_source("track", "<…>.dbm.vctracks", coordinate_context=dict(volume_uuid="<volume id>", scale_factor=1))`.
4. Lasagna: download the scroll's `*.lasagna.json` and the `nx`, `ny`, `grad_mag` arrays; the fit packs them into resident pools (`pack_resident_pools.py MANIFEST --cache DIR`).
5. Manifest `spiral-inputs.json`: `{"schema_version": 1, "volume_uuid": "<volume id>", "scroll": {"name": SCROLL, "voxel_size_um": UM, "base_shape_zyx": S["shape_zyx"], "spiral_outward_sense": "CW" or "ACW"}, "umbilicus": {"path": "umbilicus.json"}, "lasagna": {"manifest": "…lasagna.json", "original_url": "<its https URL>"}}`. The sense is not detectable from fit metrics (the 0211 and 0826 teams fitted both); the catalog flags `z_direction_is_top_to_bottom` / `left_handed_coordinates` decide it when present.
6. `FIT_SPIRAL_OUT_DIR=… FIT_SPIRAL_CONFIG_OVERRIDES='{"z_begin": Z0, "z_end": Z1, "input_use_tracks": true, "loss_weight_shell_outer": 0, "loss_weight_shell_patch_radius": 0, "optimizer_num_training_steps": 30000}' python fit_spiral.py --inputs spiral-inputs.json --object-store scroll.vcobj --cache DIR` (30-65 min on an A10 for 1,000 slices per the 0826/0211 reports).
7. Per winding: optional snap onto m7 (343's `snap_meshes.py`), then `python /content/villa/lasagna/fit.py configs/flatten_fast_nofilter.json flat.json --out-dir OUT --device cuda` with `flat.json = {"external_surfaces": [{"path": "<winding tifxyz>"}]}` → `OUT/flatten.tifxyz`; render and ink it with sections 10-13 (`MESH = …/flatten.tifxyz`).